In [ ]:
# OmniVoice's own lockfile, from PyTorch's cu128 index for torch; torchvision at
# the version that belongs to that torch so transformers can import beside it.
!pip -q install "torch==2.8.0" "torchaudio==2.8.0" "torchvision==0.23.0" --index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -3
!pip -q install "transformers==5.3.0" "accelerate==1.13.0" "webdataset==1.0.2" "safetensors==0.7.0" "tokenizers==0.22.2" "huggingface-hub==1.7.2" "tensorboardX==2.6.4" "soundfile==0.13.1" "librosa==0.11.0" "pydub==0.25.1" 2>&1 | tail -3
!pip -q install --no-deps "git+https://github.com/k2-fsa/OmniVoice@08be0b4ccbac3e13e374e86fbfead4b4cac343e2" 2>&1 | tail -3
!pip -q install speechbrain faster-whisper jiwer pyloudnorm noisereduce num2words silero-vad onnxruntime 2>&1 | tail -3


In [ ]:
import hashlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path


def captured(command):
    result = subprocess.run(command, capture_output=True, text=True)
    return result.returncode, (result.stdout + result.stderr).strip()


probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                       text=True, capture_output=True)
if probe.returncode != 0 and "torchvision" in probe.stderr:
    print("transformers cannot import beside this torchvision - removing torchvision")
    print(captured([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchvision"])[1])
    probe = subprocess.run([sys.executable, "-c", "from transformers import PreTrainedModel"],
                           text=True, capture_output=True)
assert probe.returncode == 0, probe.stderr[-2000:]

import torch

OFFICIAL = {'torch': '2.8.0+cu128', 'torchaudio': '2.8.0+cu128', 'transformers': '5.3.0', 'accelerate': '1.13.0', 'webdataset': '1.0.2', 'safetensors': '0.7.0', 'tokenizers': '0.22.2', 'huggingface-hub': '1.7.2', 'tensorboardX': '2.6.4', 'soundfile': '0.13.1', 'librosa': '0.11.0', 'pydub': '0.25.1', 'numpy': '2.4.3', 'triton': '3.4.0'}
PINNED = ('torch', 'torchaudio', 'transformers', 'accelerate', 'webdataset', 'safetensors', 'tokenizers', 'huggingface-hub')


def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


def matches(have, want):
    if have is None:
        return False
    return have == want or (have.split("+")[0] == want.split("+")[0] and "+" not in want)


stack = {name: {"official": want, "installed": installed(name),
                "matches": matches(installed(name), want), "pinned_here": name in PINNED}
         for name, want in OFFICIAL.items()}
direct = json.loads(importlib.metadata.distribution("omnivoice").read_text("direct_url.json") or "{}")
code, driver = captured(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"])
ENVIRONMENT_RECORD = {
    "python": platform.python_version(),
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "bf16_supported_including_emulation": torch.cuda.is_bf16_supported(),
    "gpus": [{"name": torch.cuda.get_device_name(i), "capability": list(torch.cuda.get_device_capability(i))}
             for i in range(torch.cuda.device_count())],
    "nvidia_smi": driver,
    "official_stack": stack,
    "omnivoice": {"version": installed("omnivoice"),
                  "commit": (direct.get("vcs_info") or {}).get("commit_id"), "url": direct.get("url")},
    "other_packages": {name: installed(name) for name in ("torchvision", "ipykernel", "papermill", "scipy")},
}
Path("/kaggle/working/environment.json").write_text(json.dumps(ENVIRONMENT_RECORD, indent=2))
for name, row in stack.items():
    mark = "ok" if row["matches"] else ("DIFFERS" if row["pinned_here"] else "differs - image's version kept")
    print(f"{name:16} official {row['official']:12} installed {str(row['installed']):16} {mark}")
print("omnivoice", ENVIRONMENT_RECORD["omnivoice"])
print("cuda", torch.version.cuda, "| cudnn", torch.backends.cudnn.version(), "|", driver)
for name in PINNED:
    assert stack[name]["matches"], f"{name} is not the official {OFFICIAL[name]}: {stack[name]['installed']}"
assert ENVIRONMENT_RECORD["omnivoice"]["commit"] == "08be0b4ccbac3e13e374e86fbfead4b4cac343e2", ENVIRONMENT_RECORD["omnivoice"]
assert torch.cuda.device_count() >= 2, "the official recipe trains on two GPUs - use T4 x2"
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i}: {torch.cuda.get_device_name(i)}, capability {torch.cuda.get_device_capability(i)}, "
          f"{free / 2**30:.1f} GiB free of {total / 2**30:.1f}")
print("disk:", f"{shutil.disk_usage('/tmp').free / 2**30:.0f} GiB free under /tmp")

INPUT = Path("/kaggle/input")
found = sorted(p for p in INPUT.rglob("repo/scripts/render_frontend_conditions.py") if "persian-voice-frontend-listening-v2" in p.parts)
if not found:
    for path in sorted(p for p in INPUT.rglob("*") if p.is_dir())[:40]:
        print("   ", path)
    raise SystemExit("no repo with scripts/render_frontend_conditions.py under /kaggle/input")
REPO = Path("/kaggle/working/repo")
if REPO.exists():
    shutil.rmtree(REPO)
shutil.copytree(found[0].parent.parent, REPO)
# Kaggle unpacks the .gz when it builds the dataset; either form is checked
# against the hash the split builder recorded for its decompressed content.
MANIFESTS = REPO / "eval/manifests"
SPLITS = next((p for p in (MANIFESTS / "adaptation_splits_v1.jsonl.gz", MANIFESTS / "adaptation_splits_v1.jsonl")
               if p.is_file()), None)
assert SPLITS is not None, "the dataset predates the adaptation splits"
import gzip  # noqa: E402

content = gzip.open(SPLITS).read() if SPLITS.suffix == ".gz" else SPLITS.read_bytes()
expected = json.loads((MANIFESTS / "adaptation_splits_v1.summary.json").read_text())["records_sha256"]
assert hashlib.sha256(content).hexdigest() == expected, "the split file is not the one the builder wrote"
print("splits:", SPLITS.name, "sha256 matches the summary")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONPATH"] = str(REPO / "src")
os.environ["VOICECLONE_SCORING_ENV"] = "kaggle:persian-voice-ezafe-check-run-v1"
sys.path.insert(0, str(REPO / "scripts"))
from kernel_gates import Gates, report_group, run_bounded  # noqa: E402

GATES = Gates()
RUNS = {}
LOGS = Path("/kaggle/working/logs")
LOGS.mkdir(exist_ok=True)
WORK = Path("/tmp/adapt")
WORK.mkdir(parents=True, exist_ok=True)
RECORDS = Path("/kaggle/working/records")
RECORDS.mkdir(exist_ok=True)


In [ ]:
from runner_check import check, diagnostics  # noqa: E402

ok, reason, run = check(run_bounded, Path("/kaggle/working/runner_check"), REPO / "scripts/moss15_worker.py")
RUNS["runner_check"] = run.record()
print(run.summary())
seen, line = diagnostics(report_group)
print("descriptor report:", "works" if seen else "DOES NOT WORK", "-", line)
GATES.record("runner", ok, reason)


In [ ]:
def fetch(plan, minutes, flags=()):
    """scripts/fetch_inputs.py under run_bounded: every request timed out, the stage bounded."""
    plan_path, report_path = RECORDS / "fetch_plan.json", RECORDS / "fetch.json"
    plan_path.write_text(json.dumps(plan))
    run = run_bounded([sys.executable, str(REPO / "scripts/fetch_inputs.py"), "--plan", str(plan_path),
                       "--report", str(report_path), *flags], timeout=minutes * 60, log=LOGS / "fetch.log",
                      monitor_every=300)
    RUNS["fetch"] = run.record()
    print(run.summary())
    report = json.loads(report_path.read_text()) if report_path.is_file() else {}
    fetched = {step["name"]: {k: step.get(k) for k in ("gib", "seconds", "tries", "failed_check")}
               for step in report.get("steps", [])}
    return run.ok and report.get("ok") is True, fetched

(REPO / "eval/runs/ezafe-check-v1/A1/texts.json").parent.mkdir(parents=True, exist_ok=True)
(REPO / "eval/runs/ezafe-check-v1/A1/texts.json").write_text('{\n "sentences_file": "eval/manifests/ezafe_check_v1.json",\n "sentences_sha256": "52b966c1cee970680215f48e6962180ca7154de06b6188517c187bafa66cc480",\n "code_commit": "861248e",\n "conditions": {\n  "A1": "the app\'s weights: the adapted export at step 1,000"\n },\n "sentences": [\n  {\n   "id": "e01",\n   "category": "news",\n   "text": "وزارت نیرو اعلام کرد مصرف برق خانگی در تابستان امسال هشت درصد کمتر از پارسال بوده است.",\n   "app_notes": [\n    "Ezafe written in on 4 word(s) by hazm\'s ParsBERT tagger: وزارت، مصرف، برق، تابستان.",\n    "First vowel written on 1 homograph(s) from Homo-GE2PE: کرد."\n   ],\n   "texts": {\n    "A1": "وزارتِ نیرو اعلام کَرد مصرفِ برقِ خانگی در تابستانِ امسال هشت درصد کمتر از پارسال بوده است."\n   }\n  },\n  {\n   "id": "e02",\n   "category": "news",\n   "text": "مأموران گمرک بندرعباس هفته گذشته محموله بزرگی از لوازم خانگی قاچاق را کشف کردند.",\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: مأموران، گمرک، هفته، محموله، لوازم، خانگی."\n   ],\n   "texts": {\n    "A1": "مأمورانِ گمرکِ بندرعباس هفتهٔ گذشته محمولهٔ بزرگی از لوازمِ خانگیِ قاچاق را کشف کردند."\n   }\n  },\n  {\n   "id": "e03",\n   "category": "news",\n   "text": "تیم ملی والیبال ایران شب گذشته در دیداری نفس\u200cگیر حریف ژاپنی خود را شکست داد.",\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: تیم، ملی، والیبال، شب، حریف، ژاپنی."\n   ],\n   "texts": {\n    "A1": "تیمِ ملیِ والیبالِ ایران شبِ گذشته در دیداری نفس\u200cگیر حریفِ ژاپنیِ خود را شکست داد."\n   }\n  },\n  {\n   "id": "e04",\n   "category": "news",\n   "text": "هفته پیش قیمت خودروهای داخلی در بازار تهران دوباره بالا رفت.",\n   "app_notes": [\n    "Ezafe written in on 4 word(s) by hazm\'s ParsBERT tagger: هفته، قیمت، خودروهای، بازار."\n   ],\n   "texts": {\n    "A1": "هفتهٔ پیش قیمتِ خودروهایِ داخلی در بازارِ تهران دوباره بالا رفت."\n   }\n  },\n  {\n   "id": "e05",\n   "category": "news",\n   "text": "سازمان هواشناسی برای روزهای پایانی هفته در استان\u200cهای شمالی باران شدید پیش\u200cبینی کرده است.",\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: سازمان، برای، روزهای، پایانی، استان\u200cهای، باران.",\n    "First vowel written on 1 homograph(s) from Homo-GE2PE: شدید."\n   ],\n   "texts": {\n    "A1": "سازمانِ هواشناسی برایِ روزهایِ پایانیِ هفته در استان\u200cهایِ شمالی بارانِ شَدید پیش\u200cبینی کرده است."\n   }\n  },\n  {\n   "id": "e06",\n   "category": "news",\n   "text": "پزشکان بیمارستان کودکان شیراز دیروز نخستین عمل پیوند قلب این استان را با موفقیت انجام دادند.",\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: پزشکان، بیمارستان، کودکان، عمل، پیوند، قلب."\n   ],\n   "texts": {\n    "A1": "پزشکانِ بیمارستانِ کودکانِ شیراز دیروز نخستین عملِ پیوندِ قلبِ این استان را با موفقیت انجام دادند."\n   }\n  },\n  {\n   "id": "e07",\n   "category": "narrative",\n   "text": "مادرم هر روز صبح قبل از رفتن به اداره گلدان\u200cهای بالکن را آب می\u200cدهد.",\n   "app_notes": [\n    "Ezafe written in on 1 word(s) by hazm\'s ParsBERT tagger: گلدان\u200cهای."\n   ],\n   "texts": {\n    "A1": "مادرم هر روز صبح قبل از رفتن به اداره گلدان\u200cهایِ بالکن را آب می\u200cدهد."\n   }\n  },\n  {\n   "id": "e08",\n   "category": "narrative",\n   "text": "پسر همسایه دیشب تا دیروقت در کوچه با دوستانش فوتبال بازی می\u200cکرد.",\n   "app_notes": [\n    "Ezafe written in on 1 word(s) by hazm\'s ParsBERT tagger: پسر."\n   ],\n   "texts": {\n    "A1": "پسرِ همسایه دیشب تا دیروقت در کوچه با دوستانش فوتبال بازی می\u200cکرد."\n   }\n  },\n  {\n   "id": "e09",\n   "category": "narrative",\n   "text": "بعد از شام پدربزرگ برای همه چای ریخت و از خاطره\u200cهای جوانی\u200cاش گفت.",\n   "app_notes": [\n    "Ezafe written in on 2 word(s) by hazm\'s ParsBERT tagger: برای، خاطره\u200cهای.",\n    "First vowel written on 1 homograph(s) from Homo-GE2PE: بعد."\n   ],\n   "texts": {\n    "A1": "بَعد از شام پدربزرگ برایِ همه چای ریخت و از خاطره\u200cهایِ جوانی\u200cاش گفت."\n   }\n  },\n  {\n   "id": "e10",\n   "category": "narrative",\n   "text": "امروز عصر خواهر کوچکم نقاشی تازه\u200cاش را با ذوق به همه نشان داد.",\n   "app_notes": [\n    "Ezafe written in on 2 word(s) by hazm\'s ParsBERT tagger: خواهر، نقاشی."\n   ],\n   "texts": {\n    "A1": "امروز عصر خواهرِ کوچکم نقاشیِ تازه\u200cاش را با ذوق به همه نشان داد."\n   }\n  },\n  {\n   "id": "e11",\n   "category": "narrative",\n   "text": "راننده اتوبوس وسط راه ایستاد و از مسافران خواست کمربندهایشان را ببندند.",\n   "app_notes": [\n    "Ezafe written in on 2 word(s) by hazm\'s ParsBERT tagger: راننده، وسط."\n   ],\n   "texts": {\n    "A1": "رانندهٔ اتوبوس وسطِ راه ایستاد و از مسافران خواست کمربندهایشان را ببندند."\n   }\n  },\n  {\n   "id": "e12",\n   "category": "narrative",\n   "text": "دوستم سال گذشته مغازه کوچکی در بازار باز کرد و حالا مشتری\u200cهای زیادی دارد.",\n   "app_notes": [\n    "Ezafe written in on 3 word(s) by hazm\'s ParsBERT tagger: سال، مغازه، مشتری\u200cهای.",\n    "First vowel written on 1 homograph(s) from Homo-GE2PE: کرد."\n   ],\n   "texts": {\n    "A1": "دوستم سالِ گذشته مغازهٔ کوچکی در بازار باز کَرد و حالا مشتری\u200cهایِ زیادی دارد."\n   }\n  }\n ]\n}\n', encoding="utf-8")
(REPO / "eval/runs/ezafe-check-v1/E1/texts.json").parent.mkdir(parents=True, exist_ok=True)
(REPO / "eval/runs/ezafe-check-v1/E1/texts.json").write_text('{\n "sentences_file": "eval/manifests/ezafe_check_v1.json",\n "sentences_sha256": "52b966c1cee970680215f48e6962180ca7154de06b6188517c187bafa66cc480",\n "code_commit": "861248e",\n "conditions": {\n  "E1": "the ezafe-marked run\'s export at step 1,000"\n },\n "sentences": [\n  {\n   "id": "e01",\n   "category": "news",\n   "text": "وزارت نیرو اعلام کرد مصرف برق خانگی در تابستان امسال هشت درصد کمتر از پارسال بوده است.",\n   "app_notes": [\n    "Ezafe written in on 4 word(s) by hazm\'s ParsBERT tagger: وزارت، مصرف، برق، تابستان.",\n    "First vowel written on 1 homograph(s) from Homo-GE2PE: کرد."\n   ],\n   "texts": {\n    "E1": "وزارتِ نیرو اعلام کَرد مصرفِ برقِ خانگی در تابستانِ امسال هشت درصد کمتر از پارسال بوده است."\n   }\n  },\n  {\n   "id": "e02",\n   "category": "news",\n   "text": "مأموران گمرک بندرعباس هفته گذشته محموله بزرگی از لوازم خانگی قاچاق را کشف کردند.",\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: مأموران، گمرک، هفته، محموله، لوازم، خانگی."\n   ],\n   "texts": {\n    "E1": "مأمورانِ گمرکِ بندرعباس هفتهٔ گذشته محمولهٔ بزرگی از لوازمِ خانگیِ قاچاق را کشف کردند."\n   }\n  },\n  {\n   "id": "e03",\n   "category": "news",\n   "text": "تیم ملی والیبال ایران شب گذشته در دیداری نفس\u200cگیر حریف ژاپنی خود را شکست داد.",\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: تیم، ملی، والیبال، شب، حریف، ژاپنی."\n   ],\n   "texts": {\n    "E1": "تیمِ ملیِ والیبالِ ایران شبِ گذشته در دیداری نفس\u200cگیر حریفِ ژاپنیِ خود را شکست داد."\n   }\n  },\n  {\n   "id": "e04",\n   "category": "news",\n   "text": "هفته پیش قیمت خودروهای داخلی در بازار تهران دوباره بالا رفت.",\n   "app_notes": [\n    "Ezafe written in on 4 word(s) by hazm\'s ParsBERT tagger: هفته، قیمت، خودروهای، بازار."\n   ],\n   "texts": {\n    "E1": "هفتهٔ پیش قیمتِ خودروهایِ داخلی در بازارِ تهران دوباره بالا رفت."\n   }\n  },\n  {\n   "id": "e05",\n   "category": "news",\n   "text": "سازمان هواشناسی برای روزهای پایانی هفته در استان\u200cهای شمالی باران شدید پیش\u200cبینی کرده است.",\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: سازمان، برای، روزهای، پایانی، استان\u200cهای، باران.",\n    "First vowel written on 1 homograph(s) from Homo-GE2PE: شدید."\n   ],\n   "texts": {\n    "E1": "سازمانِ هواشناسی برایِ روزهایِ پایانیِ هفته در استان\u200cهایِ شمالی بارانِ شَدید پیش\u200cبینی کرده است."\n   }\n  },\n  {\n   "id": "e06",\n   "category": "news",\n   "text": "پزشکان بیمارستان کودکان شیراز دیروز نخستین عمل پیوند قلب این استان را با موفقیت انجام دادند.",\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: پزشکان، بیمارستان، کودکان، عمل، پیوند، قلب."\n   ],\n   "texts": {\n    "E1": "پزشکانِ بیمارستانِ کودکانِ شیراز دیروز نخستین عملِ پیوندِ قلبِ این استان را با موفقیت انجام دادند."\n   }\n  },\n  {\n   "id": "e07",\n   "category": "narrative",\n   "text": "مادرم هر روز صبح قبل از رفتن به اداره گلدان\u200cهای بالکن را آب می\u200cدهد.",\n   "app_notes": [\n    "Ezafe written in on 1 word(s) by hazm\'s ParsBERT tagger: گلدان\u200cهای."\n   ],\n   "texts": {\n    "E1": "مادرم هر روز صبح قبل از رفتن به اداره گلدان\u200cهایِ بالکن را آب می\u200cدهد."\n   }\n  },\n  {\n   "id": "e08",\n   "category": "narrative",\n   "text": "پسر همسایه دیشب تا دیروقت در کوچه با دوستانش فوتبال بازی می\u200cکرد.",\n   "app_notes": [\n    "Ezafe written in on 1 word(s) by hazm\'s ParsBERT tagger: پسر."\n   ],\n   "texts": {\n    "E1": "پسرِ همسایه دیشب تا دیروقت در کوچه با دوستانش فوتبال بازی می\u200cکرد."\n   }\n  },\n  {\n   "id": "e09",\n   "category": "narrative",\n   "text": "بعد از شام پدربزرگ برای همه چای ریخت و از خاطره\u200cهای جوانی\u200cاش گفت.",\n   "app_notes": [\n    "Ezafe written in on 2 word(s) by hazm\'s ParsBERT tagger: برای، خاطره\u200cهای.",\n    "First vowel written on 1 homograph(s) from Homo-GE2PE: بعد."\n   ],\n   "texts": {\n    "E1": "بَعد از شام پدربزرگ برایِ همه چای ریخت و از خاطره\u200cهایِ جوانی\u200cاش گفت."\n   }\n  },\n  {\n   "id": "e10",\n   "category": "narrative",\n   "text": "امروز عصر خواهر کوچکم نقاشی تازه\u200cاش را با ذوق به همه نشان داد.",\n   "app_notes": [\n    "Ezafe written in on 2 word(s) by hazm\'s ParsBERT tagger: خواهر، نقاشی."\n   ],\n   "texts": {\n    "E1": "امروز عصر خواهرِ کوچکم نقاشیِ تازه\u200cاش را با ذوق به همه نشان داد."\n   }\n  },\n  {\n   "id": "e11",\n   "category": "narrative",\n   "text": "راننده اتوبوس وسط راه ایستاد و از مسافران خواست کمربندهایشان را ببندند.",\n   "app_notes": [\n    "Ezafe written in on 2 word(s) by hazm\'s ParsBERT tagger: راننده، وسط."\n   ],\n   "texts": {\n    "E1": "رانندهٔ اتوبوس وسطِ راه ایستاد و از مسافران خواست کمربندهایشان را ببندند."\n   }\n  },\n  {\n   "id": "e12",\n   "category": "narrative",\n   "text": "دوستم سال گذشته مغازه کوچکی در بازار باز کرد و حالا مشتری\u200cهای زیادی دارد.",\n   "app_notes": [\n    "Ezafe written in on 3 word(s) by hazm\'s ParsBERT tagger: سال، مغازه، مشتری\u200cهای.",\n    "First vowel written on 1 homograph(s) from Homo-GE2PE: کرد."\n   ],\n   "texts": {\n    "E1": "دوستم سالِ گذشته مغازهٔ کوچکی در بازار باز کَرد و حالا مشتری\u200cهایِ زیادی دارد."\n   }\n  }\n ]\n}\n', encoding="utf-8")
(REPO / "eval/runs/ezafe-check-v1/E5/texts.json").parent.mkdir(parents=True, exist_ok=True)
(REPO / "eval/runs/ezafe-check-v1/E5/texts.json").write_text('{\n "sentences_file": "eval/manifests/ezafe_check_v1.json",\n "sentences_sha256": "52b966c1cee970680215f48e6962180ca7154de06b6188517c187bafa66cc480",\n "code_commit": "861248e",\n "conditions": {\n  "E5": "the ezafe-marked run\'s export at step 5,000"\n },\n "sentences": [\n  {\n   "id": "e01",\n   "category": "news",\n   "text": "وزارت نیرو اعلام کرد مصرف برق خانگی در تابستان امسال هشت درصد کمتر از پارسال بوده است.",\n   "app_notes": [\n    "Ezafe written in on 4 word(s) by hazm\'s ParsBERT tagger: وزارت، مصرف، برق، تابستان.",\n    "First vowel written on 1 homograph(s) from Homo-GE2PE: کرد."\n   ],\n   "texts": {\n    "E5": "وزارتِ نیرو اعلام کَرد مصرفِ برقِ خانگی در تابستانِ امسال هشت درصد کمتر از پارسال بوده است."\n   }\n  },\n  {\n   "id": "e02",\n   "category": "news",\n   "text": "مأموران گمرک بندرعباس هفته گذشته محموله بزرگی از لوازم خانگی قاچاق را کشف کردند.",\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: مأموران، گمرک، هفته، محموله، لوازم، خانگی."\n   ],\n   "texts": {\n    "E5": "مأمورانِ گمرکِ بندرعباس هفتهٔ گذشته محمولهٔ بزرگی از لوازمِ خانگیِ قاچاق را کشف کردند."\n   }\n  },\n  {\n   "id": "e03",\n   "category": "news",\n   "text": "تیم ملی والیبال ایران شب گذشته در دیداری نفس\u200cگیر حریف ژاپنی خود را شکست داد.",\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: تیم، ملی، والیبال، شب، حریف، ژاپنی."\n   ],\n   "texts": {\n    "E5": "تیمِ ملیِ والیبالِ ایران شبِ گذشته در دیداری نفس\u200cگیر حریفِ ژاپنیِ خود را شکست داد."\n   }\n  },\n  {\n   "id": "e04",\n   "category": "news",\n   "text": "هفته پیش قیمت خودروهای داخلی در بازار تهران دوباره بالا رفت.",\n   "app_notes": [\n    "Ezafe written in on 4 word(s) by hazm\'s ParsBERT tagger: هفته، قیمت، خودروهای، بازار."\n   ],\n   "texts": {\n    "E5": "هفتهٔ پیش قیمتِ خودروهایِ داخلی در بازارِ تهران دوباره بالا رفت."\n   }\n  },\n  {\n   "id": "e05",\n   "category": "news",\n   "text": "سازمان هواشناسی برای روزهای پایانی هفته در استان\u200cهای شمالی باران شدید پیش\u200cبینی کرده است.",\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: سازمان، برای، روزهای، پایانی، استان\u200cهای، باران.",\n    "First vowel written on 1 homograph(s) from Homo-GE2PE: شدید."\n   ],\n   "texts": {\n    "E5": "سازمانِ هواشناسی برایِ روزهایِ پایانیِ هفته در استان\u200cهایِ شمالی بارانِ شَدید پیش\u200cبینی کرده است."\n   }\n  },\n  {\n   "id": "e06",\n   "category": "news",\n   "text": "پزشکان بیمارستان کودکان شیراز دیروز نخستین عمل پیوند قلب این استان را با موفقیت انجام دادند.",\n   "app_notes": [\n    "Ezafe written in on 6 word(s) by hazm\'s ParsBERT tagger: پزشکان، بیمارستان، کودکان، عمل، پیوند، قلب."\n   ],\n   "texts": {\n    "E5": "پزشکانِ بیمارستانِ کودکانِ شیراز دیروز نخستین عملِ پیوندِ قلبِ این استان را با موفقیت انجام دادند."\n   }\n  },\n  {\n   "id": "e07",\n   "category": "narrative",\n   "text": "مادرم هر روز صبح قبل از رفتن به اداره گلدان\u200cهای بالکن را آب می\u200cدهد.",\n   "app_notes": [\n    "Ezafe written in on 1 word(s) by hazm\'s ParsBERT tagger: گلدان\u200cهای."\n   ],\n   "texts": {\n    "E5": "مادرم هر روز صبح قبل از رفتن به اداره گلدان\u200cهایِ بالکن را آب می\u200cدهد."\n   }\n  },\n  {\n   "id": "e08",\n   "category": "narrative",\n   "text": "پسر همسایه دیشب تا دیروقت در کوچه با دوستانش فوتبال بازی می\u200cکرد.",\n   "app_notes": [\n    "Ezafe written in on 1 word(s) by hazm\'s ParsBERT tagger: پسر."\n   ],\n   "texts": {\n    "E5": "پسرِ همسایه دیشب تا دیروقت در کوچه با دوستانش فوتبال بازی می\u200cکرد."\n   }\n  },\n  {\n   "id": "e09",\n   "category": "narrative",\n   "text": "بعد از شام پدربزرگ برای همه چای ریخت و از خاطره\u200cهای جوانی\u200cاش گفت.",\n   "app_notes": [\n    "Ezafe written in on 2 word(s) by hazm\'s ParsBERT tagger: برای، خاطره\u200cهای.",\n    "First vowel written on 1 homograph(s) from Homo-GE2PE: بعد."\n   ],\n   "texts": {\n    "E5": "بَعد از شام پدربزرگ برایِ همه چای ریخت و از خاطره\u200cهایِ جوانی\u200cاش گفت."\n   }\n  },\n  {\n   "id": "e10",\n   "category": "narrative",\n   "text": "امروز عصر خواهر کوچکم نقاشی تازه\u200cاش را با ذوق به همه نشان داد.",\n   "app_notes": [\n    "Ezafe written in on 2 word(s) by hazm\'s ParsBERT tagger: خواهر، نقاشی."\n   ],\n   "texts": {\n    "E5": "امروز عصر خواهرِ کوچکم نقاشیِ تازه\u200cاش را با ذوق به همه نشان داد."\n   }\n  },\n  {\n   "id": "e11",\n   "category": "narrative",\n   "text": "راننده اتوبوس وسط راه ایستاد و از مسافران خواست کمربندهایشان را ببندند.",\n   "app_notes": [\n    "Ezafe written in on 2 word(s) by hazm\'s ParsBERT tagger: راننده، وسط."\n   ],\n   "texts": {\n    "E5": "رانندهٔ اتوبوس وسطِ راه ایستاد و از مسافران خواست کمربندهایشان را ببندند."\n   }\n  },\n  {\n   "id": "e12",\n   "category": "narrative",\n   "text": "دوستم سال گذشته مغازه کوچکی در بازار باز کرد و حالا مشتری\u200cهای زیادی دارد.",\n   "app_notes": [\n    "Ezafe written in on 3 word(s) by hazm\'s ParsBERT tagger: سال، مغازه، مشتری\u200cهای.",\n    "First vowel written on 1 homograph(s) from Homo-GE2PE: کرد."\n   ],\n   "texts": {\n    "E5": "دوستم سالِ گذشته مغازهٔ کوچکی در بازار باز کَرد و حالا مشتری\u200cهایِ زیادی دارد."\n   }\n  }\n ]\n}\n', encoding="utf-8")
os.environ["VOICECLONE_OMNIVOICE_PYTHON"] = sys.executable
provenance = found[0].parents[2] / "PROVENANCE.md"
os.environ["VOICECLONE_CODE_COMMIT"] = (provenance.read_text().split("commit ")[-1].strip(" .\n")
                                        if provenance.is_file() else "the dataset's PROVENANCE.md was not found")
NEEDED = ['scripts/render_frontend_conditions.py', 'scripts/ab_omnivoice_ipa.py', 'scripts/omnivoice_worker.py', 'src/voiceclone/text/marks_fa.py', 'eval/corpus/manifest.json', 'eval/corpus/mana_000.wav', 'eval/runs/ezafe-check-v1/A1/texts.json', 'eval/runs/ezafe-check-v1/E1/texts.json', 'eval/runs/ezafe-check-v1/E5/texts.json']
missing = [name for name in NEEDED if not (REPO / name).is_file()]
REGISTERED = {'eval/runs/ezafe-check-v1/A1/texts.json': '08c717f8ff6a973775e1cfb73fd655f86f42d9c2a4e66bbe6759457d8f98768e', 'eval/runs/ezafe-check-v1/E1/texts.json': 'b7e0e05fd1b5aaf9a35492cb757b7a18e13b54bc2692103cd9e1b64661c2f966', 'eval/runs/ezafe-check-v1/E5/texts.json': 'e33f266cf3ddc07c2a2081df3770c71bd50c933581402e7f043c0774c97ede6e'}
for name, digest in REGISTERED.items():
    if (REPO / name).is_file() and hashlib.sha256((REPO / name).read_bytes()).hexdigest() != digest:
        missing.append(f"{name} is not the registered one")
GATES.record("code", not missing, f"missing or changed: {missing}" if missing else
             "the render script, the marks and the registered texts are present; code from " +
             os.environ["VOICECLONE_CODE_COMMIT"])
if not GATES.skip("download", ("runner", "code")):
    fetched_ok, fetched = fetch({'snapshots': [{'name': 'base', 'repo': 'k2-fsa/OmniVoice', 'revision': 'c5fdb5ccb189668d56333f77ba2629f4cd7535f4', 'local_dir': '/tmp/omnivoice-base', 'sha256': {'audio_tokenizer/model.safetensors': 'fe7c5e8785e0a05833e1bfc3e002ec7f55af21e306b2e7154a448c1f54ccfb0d'}}]}, 20, ['--no-xet'])
    GATES.record("download", fetched_ok, json.dumps(fetched))
# The engine reads its weights from this override; nothing else differs from the Mac's settings.
os.environ["VOICECLONE__ENGINES__OMNIVOICE__MODEL_ID"] = "/tmp/omnivoice-base"

MODELS = {'A1': 'export-1000-float16', 'E1': 'export-1000-float16', 'E5': 'export-5000-float16'}
SESSIONS = {'A1': 'persian-voice-omnivoice-train-a', 'E1': 'persian-voice-omnivoice-ezafe-a', 'E5': 'persian-voice-omnivoice-ezafe-c'}
KNOWN_SHA256 = {'A1': '9c453a74e9eab8324c27e039b9bdc052cd8d6721cecbb20df53c919a0e6ba39a'}
WEIGHTS = {}
if not GATES.skip("weights", ("download",)):
    problems = []
    for form, name in MODELS.items():
        if name is None:
            WEIGHTS[form] = {"path": "/tmp/omnivoice-base"}
            continue
        hits = sorted(p for p in INPUT.rglob(name) if p.is_dir() and (p / "model.safetensors").is_file() and SESSIONS[form] in p.parts)
        if not hits:
            problems.append(f"{form}: no {name} with a model.safetensors under /kaggle/input")
            continue
        digest = hashlib.sha256()
        with open(hits[0] / "model.safetensors", "rb") as handle:
            for block in iter(lambda: handle.read(1 << 24), b""):
                digest.update(block)
        WEIGHTS[form] = {"path": str(hits[0]), "found": [str(p) for p in hits], "model_sha256": digest.hexdigest()}
        if form in KNOWN_SHA256 and digest.hexdigest() != KNOWN_SHA256[form]:
            problems.append(f"{form}: model.safetensors is not the one recorded before")
    GATES.record("weights", not problems, json.dumps({"weights": WEIGHTS, "problems": problems}))


In [ ]:
OUT = REPO / "eval/runs/ezafe-check-v1"
RESULT = {}
for form in MODELS:
    gate = f"renders-{form}"
    if GATES.skip(gate, ("weights",)):
        continue
    os.environ["VOICECLONE__ENGINES__OMNIVOICE__MODEL_ID"] = WEIGHTS[form]["path"]
    run = run_bounded([sys.executable, str(REPO / "scripts/render_frontend_conditions.py"), "--cer", "--texts",
                       str(OUT / form / "texts.json"), "--app-normaliser"], cwd=REPO,
                      timeout=60 * 60, log=LOGS / f"render-{form}.log", monitor_every=300)
    RUNS[gate] = run.record()
    print(run.summary())
    manifest_path = OUT / form / "manifest.json"
    manifest = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {"renders": [], "events": []}
    rendered = [r for r in manifest["renders"] if r.get("wav") and Path(r["wav"]).is_file()]
    scored = [r for r in rendered if r.get("cer") is not None]
    text = run.log.read_text(errors="replace") if run.log.is_file() else ""
    checked = [e for e in manifest["events"] if e.get("event") == "marks_checked_through_the_text_pipeline"]
    used = manifest.get("settings", {}).get("omnivoice_model_id")
    RESULT[form] = {"rendered": len(rendered), "scored": len(scored), "weights": used,
                    "worker": [line.strip() for line in text.splitlines() if "OmniVoice ready" in line][:1],
                    "mean_cer": round(sum(r["cer"] for r in scored) / len(scored), 4) if scored else None,
                    "other_changes_by_the_app": checked[0]["other_changes_by_the_app"] if checked else None}
    GATES.record(gate, run.ok and len(rendered) == 12 and len(scored) == 12 and bool(checked)
                 and used == WEIGHTS[form]["path"], json.dumps(RESULT[form], ensure_ascii=False))
os.environ["VOICECLONE__ENGINES__OMNIVOICE__MODEL_ID"] = "/tmp/omnivoice-base"


In [ ]:
destination = Path("/kaggle/working/ezafe-check-v1")
destination.mkdir(exist_ok=True)
for form in MODELS:
    (destination / form).mkdir(exist_ok=True)
    for name in ("manifest.json", "texts.json"):
        if (OUT / form / name).is_file():
            shutil.copy2(OUT / form / name, destination / form / name)
    if (OUT / form / "wavs").is_dir():
        shutil.copytree(OUT / form / "wavs", destination / form / "wavs", dirs_exist_ok=True)
(destination / "gates.json").write_text(json.dumps(GATES.summary(), indent=2))
(destination / "runs.json").write_text(json.dumps(RUNS, indent=2))
(destination / "result.json").write_text(json.dumps({"renders": RESULT, "weights": WEIGHTS}, indent=2,
                                                    ensure_ascii=False))
shutil.copy2("/kaggle/working/environment.json", destination / "environment.json")
print("gates:", json.dumps(GATES.summary(), indent=1))
